In [22]:
#Install libraries
import pyodbc
import pandas as pd
import warnings
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency
warnings.filterwarnings("ignore")

In [23]:
#Set Connection 
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=ghosh;'
    'Database=BankChurn;'
    'Trusted_Connection=yes;'
)

cursor = conn.cursor()

TEST 1: 
T-Test Hypothesis

In [24]:
#Step 1: Get the data from the database
query = """
        SELECT 
            A.Balance,
            D.Churned
        FROM [dbo].[demographic] D
        JOIN [dbo].[account] A ON D.customerId = A.customerId"""

df_t_test = pd.read_sql(query, conn)
df_t_test.head()

,Balance,Churned
0,119827.49,True
1,83807.86,False
2,159660.80,True
3,119827.49,False
4,125510.82,False


In [25]:
#Step 2: Split the data into two groups based on churn status
churned_t_test = df_t_test[df_t_test['Churned'] == 1]['Balance']
retained_t_test = df_t_test[df_t_test['Churned'] == 0]['Balance']

In [26]:
#Step 3: Descriptive statistics for each group
print("Churned Mean Balance:", churned_t_test.mean())
print("Retained Mean Balance:", retained_t_test.mean())

Churned Mean Balance: 120521.27620520373
Retained Mean Balance: 119650.01690066556


In [27]:
#If p > 0.05, equal variances are assumed. If p < 0.05, equal variances are not assumed.
#Step 4: Perform Levene's test for equal variances
stats.levene(churned_t_test, retained_t_test)

LeveneResult(statistic=np.float64(36.69675448689501), pvalue=np.float64(1.4298234841363054e-09))

Since the p-value is less than 0.05, the variances are not equal which should have been the case to apply the standard T-test. Instead, we should use the Welsch's T-Test.

In [28]:
#Step 4: Perform the t-test
t_stat, p_value = stats.ttest_ind(churned_t_test, retained_t_test, equal_var=False)  
print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: 1.3534724523393942
P-value: 0.17601045694361078


The p-value is more than 0.05. It means we cannot reject the Null Hypothesis of Test. So, final interpretation is that There is no difference between the balance amount for the churned and retained prople. It shows that the churning of clients is not caused by the amount of Balance.


TEST 2: Chi-Sq Test

In [29]:
#Step 1: Get the data from the database for Chi-Square test
query = """
        SELECT 
            A.IsActive,
            D.Churned
        FROM [dbo].[demographic] D
        JOIN [dbo].[account] A ON D.customerId = A.customerId"""

df_chi = pd.read_sql(query, conn)
df_chi.head()

,IsActive,Churned
0,True,True
1,True,False
2,False,True
3,False,False
4,True,False


In [30]:
#Step 2: Create a contingency table
ct = pd.crosstab(df_chi['IsActive'], df_chi['Churned'])
print(ct)

Churned   False  True 
IsActive              
False      3547   1302
True       4416    735


In [31]:
#Step 3: Run the Chi-Square test
chi2, p, dof, expected = chi2_contingency(ct)

print("Chi-Square Statistic:", chi2)
print("P-value:", p)    
print("Degrees of Freedom:", dof)

Chi-Square Statistic: 242.98534164287963
P-value: 8.785858269303703e-55
Degrees of Freedom: 1


In [32]:
#Step 4: Expected frequencies
print("Expected Frequencies:\n", expected)

Expected Frequencies:
 [[3861.2587  987.7413]
 [4101.7413 1049.2587]]


OUTPUT INTERPRETATION:
the p-value is less than 0.05. It means we reject the null hypothesis which in case of the Chi-Square test is that "Churn and Customer Activity are independent". So, the final interpretation is that Churn and Customer Activity are associated(dependent).

TEST 3: ANOVA Test

In [33]:
#Step 1: Get the data from the database for ANOVA test
query = """
        SELECT
            A.Tenure,
            L.Geography Country
        FROM [dbo].[demographic] D
        JOIN [dbo].[account] A ON D.CustomerId = A.CustomerId
        JOIN [dbo].[location] L ON D.LocationId = L.LocationId
        WHERE A.Tenure IS NOT NULL AND L.Geography IS NOT NULL"""

df_anova = pd.read_sql(query, conn)
df_anova.head()


,Tenure,Country
0,9,Spain
1,1,Germany
2,1,USA
3,6,Spain
4,10,Spain


In [34]:
#Step 2: Group the data by Geography and get the Tenure values for each group
grouped = df_anova.groupby('Country')['Tenure'].aggregate(count='count', mean='mean', std='std')
print(grouped.sort_values(by='mean', ascending=False))
                                                        

         count      mean       std
Country                           
UK        1703  5.127422  2.939195
France    1714  5.042590  2.904582
USA       1669  5.002397  2.890694
Germany   1626  4.987700  2.870085
Canada    1679  4.982132  2.888003
Spain     1609  4.927906  2.857046


In [35]:
#Step 3: Check Variance Homogeneity using Levene's Test


# Extract the Tenure values for each country
tenure_by_country = [group['Tenure'].values for name, group in df_anova.groupby('Country')]

# Perform Levene's test
stat, p_value = stats.levene(*tenure_by_country)
print(f"Levene's Test - Statistic: {stat}, P-Value: {p_value}")

Levene's Test - Statistic: 0.5338267546333629, P-Value: 0.7508283480014718


In [36]:
#Step 4: Perform ANOVA test
anova_stat, anova_p_value = stats.f_oneway(*tenure_by_country)
print(f"ANOVA Test - Statistic: {anova_stat}, P-Value: {anova_p_value}")

ANOVA Test - Statistic: 0.9151410065439705, P-Value: 0.4698663379101742


Cohen's d: T-test

In [37]:
#Cohen's d calculation for effect size 

#means
churned_mean = churned_t_test.mean()
retained_mean = retained_t_test.mean()

#standard deviations
churned_std = churned_t_test.std()
retained_std = retained_t_test.std()

#sample sizes
churned_n = len(churned_t_test)
retained_n = len(retained_t_test)

#pooled standard deviation
pooled_std = np.sqrt(((churned_n - 1) * churned_std ** 2 + (retained_n - 1) * retained_std ** 2) / (churned_n + retained_n - 2))

#Cohen's d calculation
cohens_d = (churned_mean - retained_mean) / pooled_std
print("Cohen's d:", cohens_d)

Cohen's d: 0.03623904081079434


The Independent Samples t-test indicated no statistically significant difference in average account balance between churned and retained customers (high p-value). Additionally, the effect size (Cohen’s d = 0.03) is negligible, suggesting that balance has minimal practical impact on churn behavior.

Cramer's V: Chi-Square

In [38]:
#Cramer's V calculation for effect size

n_chi = ct.sum().sum()  # Total number of observations
k_chi = min(ct.shape) 

cramers_v = np.sqrt(chi2 / (n_chi * (k_chi - 1)))
print("Cramer's V:", cramers_v)

Cramer's V: 0.15587987094005423


The Chi-Square test shows a statistically significant relationship between customer activity and churn. However, Cramér’s V = 0.15 indicates a weak association, suggesting that while engagement is related to churn, it is not the sole driving factor.

Eta/Omega: ANOVA


In [43]:
#Calculate SS
from tokenize import group


all_values = df_anova["Tenure"].values
grand_mean = np.mean(all_values)

ss_between = sum(len(group) * (np.mean(group) - grand_mean)**2 for group in tenure_by_country)

ss_within = sum(
    sum((group - np.mean(group))**2)
    for group in tenure_by_country
)

ss_total = ss_between + ss_within
print("SS Total:", ss_total)

SS Total: 83638.36159999996


In [44]:
#Calculate ETA score
k = len(tenure_by_country)  # Number of groups
n = len(all_values)  # Total number of observations

df_between = k - 1
df_within = n - k

ms_within = ss_within / df_within

eta_squared = ss_between / ss_total
print("Eta Squared:", eta_squared)

Eta Squared: 0.00045763568409215794


In [45]:
#Calculate Omega Squared
omega_squared = (ss_between - (df_between * ms_within)) / (ss_total + ms_within)

print("Omega Squared:", omega_squared)

Omega Squared: -4.2431297066592226e-05


The one-way ANOVA test indicates no statistically significant difference in average tenure across countries (p-value > 0.05). Effect size measures further confirm this finding, with Eta Squared (0.00046) and Omega Squared (0) indicating a negligible practical impact. This suggests that geographic region does not meaningfully influence customer tenure in this dataset.